# freezing here

In [1]:
from src.MPOptoClass import *
from src.utils.gen_utils import *
from src.utils.filters import *
from src.helpers.experiment import *
from src.wiener_filter import *
from src.modeller import *

import copy
import time
import mat73
import pynapple as nap
import numpy as np
import matplotlib.pyplot as plt
import random
from sklearn.decomposition import PCA
from itertools import permutations, compress



%load_ext autoreload 
%autoreload 2
%matplotlib widget

# lets first look at fit scores only, nothing really held out

In [2]:
pre=100
post=100
session_path = '../data/vgat2_06062023'
session = MPOptoClass(session_path, post=post)

(all_nlags_PCA, all_cut_PCA), (CFA_PCObj, RFA_PCObj) = session.format_nlags_PCA(bounds=session.climbing_bounds, binsize=10, nlags=10)


love


In [3]:
psth_bounds = session.get_highlaser_bounds()
psth_ctrl_bounds = session.laser['ctrl_bounds']
psth_ctrl_bounds[:,0] = psth_ctrl_bounds[:,0] - pre
psth_bounds[:,0] = psth_bounds[:,0] - pre
psth_newbounds, climbing_duration = reBoundInBounds(session.climbing_bounds, psth_bounds)
psth_logical = bounds2Logical(psth_newbounds, duration=climbing_duration)
psth_logical_trials = unstitchSeams(psth_logical, getSeamsFromBounds(session.climbing_bounds, binsize=1))

sw_list = []
for trial in psth_logical_trials:
    temp = bin_timeseries(trial, binsize=10)
    sw_list.append(format_single_array(temp))
sw = (np.hstack(sw_list))[:-1]

sw = (sw*4)+1
control_sw = np.ones(sw.shape[0])

In [ ]:
h_laser = weighted_parameter_fit(all_nlags_PCA[:50000,:], all_cut_PCA[:50000,:], c=50000, sw=sw[:50000])
h_ctrl = weighted_parameter_fit(all_nlags_PCA[:50000,:], all_cut_PCA[:50000,:], c=50000, sw=control_sw[:50000])

In [ ]:
binsize=10
CFA_lt, RFA_lt = session.binner(bounds=psth_bounds, fs=10, concat=True)
CFA_lt_PCA = CFA_PCObj.transform(CFA_lt)
RFA_lt_PCA = RFA_PCObj.transform(RFA_lt)
all_lt_PCA = np.hstack((CFA_lt_PCA, RFA_lt_PCA))
all_lt_PCA = unstitchSeams(all_lt_PCA, getSeamsFromBounds(bounds=psth_bounds, binsize=binsize))

all_laser_input, all_laser_true = format_and_stitch(all_lt_PCA, all_lt_PCA)
all_laser_input = all_laser_input[:-1, :]
all_laser_true = all_laser_true[1:,:]

CFA_clt, RFA_clt = session.binner(bounds=psth_ctrl_bounds, fs=binsize, concat=True)
CFA_clt_PCA = CFA_PCObj.transform(CFA_clt)
RFA_clt_PCA = RFA_PCObj.transform(RFA_clt)
all_clt_PCA = np.hstack((CFA_clt_PCA, RFA_clt_PCA))
all_clt_PCA = unstitchSeams(all_clt_PCA, getSeamsFromBounds(bounds=psth_ctrl_bounds, binsize=binsize))

all_c_laser_input, all_c_laser_true = format_and_stitch(all_clt_PCA, all_clt_PCA)
all_c_laser_input = all_c_laser_input[:-1, :]
all_c_laser_true = all_c_laser_true[1:,:]

In [ ]:
temp1 = test_wiener_filter(all_laser_input, h_laser)
temp2 = test_wiener_filter(all_nlags_PCA, h_laser)

laser_true = all_laser_true[:, session.num_CFA:]
laser_predic = temp1[:, session.num_CFA:]
laser_r2 = weighted_r2(laser_true, laser_predic)

ctrl_true = all_cut_PCA[:, session.num_CFA:]
ctrl_predic = temp2[:, session.num_CFA:]
ctrl_r2 = weighted_r2(ctrl_true, ctrl_predic)

temp1 = test_wiener_filter(all_laser_input, h_ctrl)
temp2 = test_wiener_filter(all_nlags_PCA, h_ctrl)

claser_true = all_laser_true[:, session.num_CFA:]
claser_predic = temp1[:, session.num_CFA:]
claser_r2 = weighted_r2(claser_true, claser_predic)

cctrl_true = all_cut_PCA[:, session.num_CFA:]
cctrl_predic = temp2[:, session.num_CFA:]
cctrl_r2 = weighted_r2(cctrl_true, cctrl_predic)



In [ ]:
fig, ax = plt.subplots(nrows=2)
label1 = ['weighted_fit', 'weighted_laser']
label2 = ['normal_fit', 'normal_laser']
p = ax[0].bar(label1, [ctrl_r2, laser_r2])
ax[0].bar_label(p, label_type='center')
p2 = ax[1].bar(label2, [cctrl_r2, claser_r2])
ax[1].bar_label(p2, label_type='center')
ax[0].set_ylabel('r2')
ax[0].set_title('only fit scores, nothing held out')

# okay lets hold out a couple laser trials, and ctrl trials, and see how the test sets do

In [ ]:
post=100
pre=100
session_path = '../data/vgat2_06062023'
session = MPOptoClass(session_path, post=post)

psth_bounds = session.get_highlaser_bounds()
psth_bounds[:,0] = psth_bounds[:,0] - pre
psth_ctrl_bounds = copy.deepcopy(session.laser['ctrl_bounds'])
psth_ctrl_bounds[:,0] = psth_ctrl_bounds[:,0] - pre

#lets hold out 8 random laser trials

total_laser_trials = np.arange(psth_bounds.shape[0]).tolist()
holds = random.sample(total_laser_trials, 8)
holds.sort()
keeps =  [trial for trial in total_laser_trials if trial not in holds]

psth_train_bounds = psth_bounds[keeps, :]
psth_test_bounds = psth_bounds[holds, :]

In [ ]:
omitLaserBounds = omitBoundInBounds(session.climbing_bounds, psth_test_bounds)
omitLaserBounds = omitBoundInBounds(omitLaserBounds, psth_ctrl_bounds)
#lets omit test lasers and ctrl lasers

psth_train_logical = session.climbing_logical + bounds2Logical(psth_train_bounds, session.climbing_logical.shape) #we need this to identify where the train samples are
psth_test_logical = bounds2Logical(psth_test_bounds, session.climbing_logical.shape)
psth_ctrl_logical = bounds2Logical(psth_ctrl_bounds, session.climbing_logical.shape)

temp = psth_train_logical - psth_test_logical - psth_ctrl_logical #it'll be 0 or negative when not climbing, nor not in test/ctrl lasers
psth_train_positions = temp[temp >= 1] - 1
#psth_train_positions_binned = (bin_timeseries(psth_train_positions, binsize=10) > 5) * 1

psth_train_positions_trials = unstitchSeams(psth_train_positions, getSeamsFromBounds(omitLaserBounds, binsize=1))


#i think this works but likely needs to be chekced


In [ ]:
psth_train_positions_final = []
for trial in psth_train_positions_trials:
    temp = bin_timeseries(trial, binsize=10)
    psth_train_positions_final.append(format_single_array(temp))
    
psth_train_positions_final = (np.hstack(psth_train_positions_final))[:-1]

In [ ]:
(all_nlags_PCA, all_cut_PCA), (CFA_PCObj, RFA_PCObj) = session.format_nlags_PCA(bounds=omitLaserBounds, binsize=10, nlags=10)


In [ ]:
weights = copy.deepcopy(psth_train_positions_final)
weights = (weights*5)+1
ctrl_weights = np.ones(weights.size)
print(weights.shape)

In [ ]:
h_w=weighted_parameter_fit(all_nlags_PCA[:40000,:], all_cut_PCA[:40000,:], c=50000, sw=weights[:40000])
h_c= weighted_parameter_fit(all_nlags_PCA[:40000,:], all_cut_PCA[:40000,:], c=50000, sw=ctrl_weights[:40000])

In [ ]:
binsize=10
CFA_lt, RFA_lt = session.binner(bounds=psth_test_bounds, fs=10, concat=True)
CFA_lt_PCA = CFA_PCObj.transform(CFA_lt)
RFA_lt_PCA = RFA_PCObj.transform(RFA_lt)
all_lt_PCA = np.hstack((CFA_lt_PCA, RFA_lt_PCA))
all_lt_PCA = unstitchSeams(all_lt_PCA, getSeamsFromBounds(bounds=psth_test_bounds, binsize=binsize))

all_laser_input, all_laser_true = format_and_stitch(all_lt_PCA, all_lt_PCA)
all_laser_input = all_laser_input[:-1, :]
all_laser_true = all_laser_true[1:,:]

CFA_clt, RFA_clt = session.binner(bounds=psth_ctrl_bounds, fs=binsize, concat=True)
CFA_clt_PCA = CFA_PCObj.transform(CFA_clt)
RFA_clt_PCA = RFA_PCObj.transform(RFA_clt)
all_clt_PCA = np.hstack((CFA_clt_PCA, RFA_clt_PCA))
all_clt_PCA = unstitchSeams(all_clt_PCA, getSeamsFromBounds(bounds=psth_ctrl_bounds, binsize=binsize))

all_c_laser_input, all_c_laser_true = format_and_stitch(all_clt_PCA, all_clt_PCA)
all_c_laser_input = all_c_laser_input[:-1, :]
all_c_laser_true = all_c_laser_true[1:,:]

In [ ]:
temp1 = test_wiener_filter(all_laser_input, h_w)
temp2 = test_wiener_filter(all_c_laser_input, h_w)

laser_true = all_laser_true[:, session.num_CFA:]
laser_predic = temp1[:, session.num_CFA:]
laser_r2 = weighted_r2(laser_true, laser_predic)

ctrl_true = all_c_laser_true[:, session.num_CFA:]
ctrl_predic = temp2[:, session.num_CFA:]
ctrl_r2 = weighted_r2(ctrl_true, ctrl_predic)

temp1 = test_wiener_filter(all_laser_input, h_c)
temp2 = test_wiener_filter(all_c_laser_input, h_c)

claser_true = all_laser_true[:, session.num_CFA:]
claser_predic = temp1[:, session.num_CFA:]
claser_r2 = weighted_r2(claser_true, claser_predic)

cctrl_true = all_c_laser_true[:, session.num_CFA:]
cctrl_predic = temp2[:, session.num_CFA:]
cctrl_r2 = weighted_r2(cctrl_true, cctrl_predic)

In [ ]:
fig2, ax2 = plt.subplots(nrows=2)
label1 = ['weighted_fit', 'weighted_laser']
label2 = ['normal_fit', 'normal_laser']
p = ax2[0].bar(label1, [ctrl_r2, laser_r2])
ax2[0].bar_label(p, label_type='center')
p2 = ax2[1].bar(label2, [cctrl_r2, claser_r2])
ax2[1].bar_label(p2, label_type='center')
ax2[0].set_ylabel('r2')